# 03 · Nested spatial cross-validation: five models × two training extents

**Brief sections 6, 8, 9 and 13.**

| | |
|---|---|
| Models | RF, XGBoost, MLP, GWRF, GWXGB (`src/siemreap/models.py`, `gw.py`), plus a depth-band prior as a non-competing reference |
| Inputs | exactly `UTM_Easting_m`, `UTM_Northing_m`, `Top_Elevation_m` |
| Outer folds | the R blockCV folds (100, 200, 300 m); identical for every model and both scenarios |
| Scenario A | train on Downtown development boreholes outside the test fold |
| Scenario B | the same plus every 5 km ring development borehole (train-only, fold 0) |
| Test set | always the Downtown development boreholes of the outer fold, the same in A and B |
| Inner folds | the four remaining outer folds (adequacy checked in notebook 02; R fallback otherwise). Ring boreholes are always inner-training |
| Tuning | Optuna TPE with a fixed seed, maximising pooled inner out-of-fold Macro-F1. The decision threshold is chosen on the same inner OOF predictions, so no outer label is ever used for tuning, thresholds or preprocessing |
| Preprocessing | MLP scaling and balanced class weights are computed inside `fit`, i.e. on the training partition only |
| Determinism | seeds derived from (scenario, block, model, fold); single-threaded learners inside the GW models |
| Checkpoints | one file per outer fold in `outputs/03_nested_cv/checkpoints/`; re-running skips finished folds |
| Purge | separate runs on the primary block size that also drop training boreholes within 100 / 200 m of any test borehole |

**Geographically weighted models.** Every training borehole is an anchor with a local model. The local model is trained on the cells of the *k* nearest training boreholes, with *k* tuned and optionally bisquare distance weights. A neighbourhood needs at least 5 boreholes and 20 cells of each class. Otherwise it grows, up to 80 boreholes, and finally falls back to the global model. A prediction uses the nearest anchor's local model blended with a global model: *p* = α *p*<sub>local</sub> + (1 − α) *p*<sub>global</sub>. This follows Georganos et al. (2021) and Grekousis (2025), adapted from regression to classification.

**Search spaces and budgets** are in `config/project.toml` (`[search_space.*]`, `[experiment.n_trials]`) and are printed below. In smoke mode the budgets are reduced (`[smoke]`) and the outputs are labelled as a software test.

In [ ]:
import os, sys, json
from pathlib import Path

# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner())

from siemreap import nested_cv as NCV
OUT = config.stage_dir(cfg, "03_nested_cv", "")
AGG = config.stage_dir(cfg, "03_nested_cv", "aggregate")
RES = config.stage_dir(cfg, "03_nested_cv", "restricted")
mt = pd.read_csv(config.processed_dir(cfg) / "cells_all.csv")
print(json.dumps({m: cfg["search_space"][m] for m in cfg["experiment"]["models"]}, indent=1)[:3000])
print("trials per outer fold:", cfg["smoke"]["n_trials"] if MODE == "smoke" else cfg["experiment"]["n_trials"])

## Run specification

In [ ]:
specs = NCV.specs_from_config(cfg, MODE)
outer_folds = cfg["smoke"]["outer_folds"] if MODE == "smoke" else list(range(1, cfg["blockcv"]["k"] + 1))

# Optional subset, e.g. to run the primary block size first or to split the work across sessions:
RUN_BLOCKS = None      # e.g. [100]
RUN_MODELS = None      # e.g. ["RF", "XGB"]
RUN_SCENARIOS = None   # e.g. ["A"]
specs = [s for s in specs if (RUN_BLOCKS is None or s.block_m in RUN_BLOCKS)
         and (RUN_MODELS is None or s.model in RUN_MODELS) and (RUN_SCENARIOS is None or s.scenario in RUN_SCENARIOS)]
print(len(specs), "runs x", len(outer_folds), "outer folds")
display(pd.DataFrame([s.__dict__ for s in specs]).groupby(["block_m", "purge_m"]).size().rename("runs"))

## Run (resumable)

In [ ]:
import time
save_models = True    # outer-fold models are saved for the primary block size only (see below)
all_preds, all_info = [], []
t0 = time.time()
for spec in specs:
    keep = save_models and spec.block_m == (cfg["smoke"]["block_sizes_m"][0] if MODE == "smoke" else cfg["blockcv"]["primary_block_m"]) and spec.purge_m == 0
    p, info = NCV.run(mt, cfg, spec, outer_folds, MODE, OUT, save_models=keep)
    all_preds.append(p); all_info += info
print(f"done in {(time.time() - t0) / 60:.1f} min")

## Save predictions and run records

In [ ]:
preds = pd.concat(all_preds, ignore_index=True)
info = pd.DataFrame(all_info)
preds.to_csv(RES / "outer_predictions.csv", index=False)            # borehole codes and coordinates: restricted
info_public = info.drop(columns=["inner_adequacy"])
info_public.to_csv(AGG / "run_info.csv", index=False)
(AGG / "environment.json").write_text(json.dumps(config.environment_record(), indent=1))
display(info_public.groupby(["block_m", "purge_m", "scenario", "model"])[["inner_macro_f1", "n_trials", "seconds"]]
        .mean().round(3))
print("inner-fold source:", info.inner_source.value_counts().to_dict())
if "gw_fallback_rate" in info:
    display(info.dropna(subset=["gw_fallback_rate"]).groupby(["scenario", "model"]).gw_fallback_rate.describe())

## Determinism check

This re-fits one outer fold of RF and XGBoost from scratch (no checkpoint) and compares the predictions with the stored ones. They must be identical.

In [ ]:
from siemreap import folds as F
spec0 = [s for s in specs if s.model in ("RF", "XGB") and s.purge_m == 0][:2]
for s in spec0:
    k = outer_folds[0]
    again = NCV.run_fold(mt, F.load_outer(cfg, s.block_m), s, k, cfg, MODE)["predictions"]
    stored = preds[(preds.model == s.model) & (preds.scenario == s.scenario) & (preds.block_m == s.block_m)
                   & (preds.purge_m == 0) & (preds.outer_fold == k)]
    same = np.array_equal(again.p_clay.values, stored.p_clay.values)
    print(s.tag, "fold", k, "identical predictions:", same)
    assert same